In [ ]:
# Directory holding the exported CLAP text-embedding CSVs (EmbedText*.csv).
folder_path = 'data/clap_embeddings'


In [ ]:
# pip ionstall cudf

In [ ]:
import os
import pandas as pd
import numpy as np

# List to store vectors from all CSVs
all_vectors = []

# Limit the number of files processed for testing
max_files = 495

# Loop through files in the folder
for Num, file in enumerate(os.listdir(folder_path)):
    vector_concat = []
    if file.endswith('.csv') and Num < max_files:
        file_path = os.path.join(folder_path, file)

        # Read CSV file into a Pandas DataFrame
        df = pd.read_csv(file_path)
        print(Num)
        # Iterate through rows and concatenate values from all columns to form vectors
        for _, row in df.iterrows():
            # Concatenate values from all columns to form a vector
            vector_values = row.values.astype(np.float16)[:1 * 1024]  # Convert to float32
            vector = vector_values.reshape((1, 1024))  # Reshape to the desired shape (1024, 4)
            # Append the vector to the list
            vector_concat.append(vector)

        all_vectors += vector_concat

# Print the first vector from the list for verification
print(all_vectors[0])


In [ ]:
print(all_vectors[0])

In [ ]:
# import numpy as np

# # Generate random data points
# num_data_points = 32
# data_size = (4, 1024)
# all_vectors = []

# for _ in range(num_data_points):
#     data_point = np.random.randn(*data_size)  # Generate random data of size (4, 1024)
#     all_vectors.append(data_point)

# # Convert the list to a numpy array if needed
# all_vectors = np.array(all_vectors)

# # Check the shape of the generated data
# print(all_vectors.shape)

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Check if GPU is available and set device accordingly
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Assuming all_vectors is a list of data points where each data point is a numpy array of shape (4, 1024)
# Also assuming folder_path is the specified path for saving the model

# Convert the list of data points to a numpy array and flatten it
data_points = np.array(all_vectors)
# Assuming the input data shape is (num_samples, 4, 1024)
tensor_data = torch.tensor(data_points, dtype=torch.float32).to(device)

# Define a Convolutional Autoencoder model with LayerNorm
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super(ConvAutoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),  # Input shape: (1, 4, 1024), Output shape: (16, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([16, 1, 1024]),  # LayerNorm after first Conv2d layer
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),  # Input shape: (16, 4, 1024), Output shape: (32, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([32, 1, 1024]),  # LayerNorm after second Conv2d layer
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # Input shape: (32, 4, 1024), Output shape: (64, 4, 1024)
            nn.ReLU(),
            nn.LayerNorm([64, 1, 1024]),  # LayerNorm after third Conv2d layer
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),  # Input shape: (64, 4, 1024), Output shape: (128, 4, 1024)
            nn.ReLU(),
            nn.Flatten(),  # Flatten the output
            nn.Linear(128 * 1* 1024, 512),

        )
        self.decoder = nn.Sequential(
            nn.Linear(512, 128 * 4 * 8),  # Adjust size to match encoder output
            nn.ReLU(),
            nn.Unflatten(dim=1, unflattened_size=(128, 4, 8)),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 4, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Conv2d(4, 1, kernel_size=3, stride=1, padding=1),
            nn.Flatten(),
            nn.Linear(64 * 128, 1 * 1 * 1024),
            nn.Unflatten(dim=1, unflattened_size=(1, 1, 1024)),  # Adjust the linear layer to match the reshaped size
            nn.Sigmoid()  # Sigmoid activation for reconstruction
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Instantiate the Convolutional Autoencoder model and move it to GPU
autoencoder = ConvAutoencoder().to(device)

# Check if saved model files exist and load them
encoder_path = folder_path + '/encoder_model.pth'
model_path = folder_path + '/model.pth'

if os.path.exists(encoder_path) and os.path.exists(model_path):
    autoencoder.load_state_dict(torch.load(model_path))
    print("Saved model loaded successfully.")

# Define loss function (reconstruction loss) and optimizer
criterion = nn.MSELoss()
optimizer = AdamW(autoencoder.parameters(), lr=1e-3)  # AdamW optimizer with lr=1e-6
scheduler = CosineAnnealingLR(optimizer, T_max=20)  # CosineAnnealingLR scheduler with T_max=20

# Create a DataLoader for batch training
batch_size = 32
dataset = TensorDataset(tensor_data)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Training loop
num_epochs = 100  # Increased to 100 epochs
min_loss = float('inf')  # Initialize min_loss with infinity

for epoch in range(num_epochs):
    running_loss = 0.0
    for data in dataloader:
        inputs = data[0].unsqueeze(1).to(device)  # Add a channel dimension and move to GPU
        optimizer.zero_grad()
        outputs = autoencoder(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    epoch_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}, Loss: {epoch_loss}")

    # Save the model if the current epoch's loss is lower than the previous min_loss
    if epoch_loss < min_loss:
        min_loss = epoch_loss
        torch.save(autoencoder.state_dict(), model_path)
        torch.save(autoencoder.encoder.state_dict(), encoder_path)

# Save only the encoder's state dictionary
torch.save(autoencoder.encoder.state_dict(), encoder_path)


In [ ]:
print(len(dataloader))

In [ ]:
def rmse(predictions, targets):
    """
    Calculate the Root Mean Squared Error (RMSE) between two vectors.

    Args:
    - predictions: numpy array, predicted values
    - targets: numpy array, actual values

    Returns:
    - rmse_value: float, RMSE between predictions and targets
    """
    rmse_value = np.sqrt(np.mean((predictions - targets)**2))
    return rmse_value
device='cpu'
for i in range(100):
  L=np.array(tensor_data[i].reshape(1024).to(device))
  K=tensor_data[i].reshape(1,1,1,1024)
  M=autoencoder.forward(K).reshape(1024).cpu().detach().numpy()
  print(L.shape,M.shape)
  print(rmse(L,M))